# 26. Refined Metric Region Policy

This notebook refines the metric-region policy used for the final three-model comparison.

The initial three-model comparison used sparse `masked_region` values for MSE, PSNR, and SSIM. However, sparse masked-region SSIM produced invalid local comparison values.

SSIM is retained as an important structural-similarity metric, but its local comparison region is revised from sparse `masked_region` to `mask_bbox_crop`.

The refined final local comparison policy is:

- MSE improvement: `masked_region`
- PSNR improvement: `masked_region`
- SSIM improvement: `mask_bbox_crop`
- LPIPS improvement: `mask_bbox_crop`
- CLIP similarity improvement: `mask_bbox_crop`
- DINOv2 similarity improvement: `mask_bbox_crop`

This preserves SSIM while aligning it with an image-like local region that contains spatial context around the damaged area.

## Notebook goal

This notebook does not regenerate model-level metrics or model-level reports.

It reuses the existing metric CSV files for:

- OpenCV Telea,
- LaMa,
- Stable Diffusion Inpainting.

It rebuilds only the comparison-level outputs using the refined metric-region policy.

The expected outputs are:

- refined unified comparison table,
- refined win-rate table,
- refined mask-type summary,
- refined category summary,
- refined metric-disagreement cases,
- old-vs-refined comparison summary,
- compact refined HTML report.

In [1]:
from pathlib import Path
import sys
import base64
from datetime import datetime

import pandas as pd
import yaml
from PIL import Image
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Project root:", PROJECT_ROOT)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval


In [2]:
config_path = PROJECT_ROOT / "config" / "experiment_50_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}")

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

paths_cfg = config["paths"]

processed_metadata_dir = PROJECT_ROOT / paths_cfg["processed_metadata_dir"]
metrics_dir = PROJECT_ROOT / paths_cfg["metrics_dir"]
reports_dir = PROJECT_ROOT / paths_cfg.get("reports_dir", "outputs/reports")

metrics_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

refined_report_path = reports_dir / "opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html"

output_paths = {
    "refined_unified": metrics_dir / "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_win_rates": metrics_dir / "comparison_win_rates_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_summary_by_mask_type": metrics_dir / "comparison_summary_by_mask_type_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_summary_by_category": metrics_dir / "comparison_summary_by_category_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_metric_disagreement_cases": metrics_dir / "comparison_metric_disagreement_cases_refined_opencv_lama_stable_diffusion_50.csv",
    "old_vs_refined": metrics_dir / "comparison_old_vs_refined_metric_policy_50.csv",
}

old_comparison_path = metrics_dir / "comparison_unified_opencv_lama_stable_diffusion_50.csv"
old_win_rates_path = metrics_dir / "comparison_win_rates_opencv_lama_stable_diffusion_50.csv"

print("Metrics dir:", metrics_dir)
print("Reports dir:", reports_dir)
print("Refined report path:", refined_report_path)

Metrics dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics
Reports dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports
Refined report path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html


In [3]:
model_registry = {
    "opencv_telea": {
        "display_name": "OpenCV Telea",
        "classical_metrics_path": metrics_dir / "classical_metrics_opencv_telea_50.csv",
        "lpips_metrics_path": metrics_dir / "lpips_metrics_opencv_telea_50.csv",
        "feature_metrics_path": metrics_dir / "feature_similarity_opencv_telea_50.csv",
    },
    "lama": {
        "display_name": "LaMa",
        "classical_metrics_path": metrics_dir / "classical_metrics_lama_50.csv",
        "lpips_metrics_path": metrics_dir / "lpips_metrics_lama_50.csv",
        "feature_metrics_path": metrics_dir / "feature_similarity_lama_50.csv",
    },
    "stable_diffusion_inpainting": {
        "display_name": "Stable Diffusion Inpainting",
        "classical_metrics_path": metrics_dir / "classical_metrics_stable_diffusion_50.csv",
        "lpips_metrics_path": metrics_dir / "lpips_metrics_stable_diffusion_50.csv",
        "feature_metrics_path": metrics_dir / "feature_similarity_stable_diffusion_50.csv",
    },
}

for model_name, model_info in model_registry.items():
    print("\nModel:", model_name)

    for key, path in model_info.items():
        if key == "display_name":
            continue

        print(f"  {key}: {path}")

        if not path.exists():
            raise FileNotFoundError(
                f"Missing input for {model_name}, {key}: {path}"
            )

if not old_comparison_path.exists():
    raise FileNotFoundError(f"Missing previous comparison file: {old_comparison_path}")

if not old_win_rates_path.exists():
    raise FileNotFoundError(f"Missing previous win-rate file: {old_win_rates_path}")

print("\nAll required inputs exist.")


Model: opencv_telea
  classical_metrics_path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\classical_metrics_opencv_telea_50.csv
  lpips_metrics_path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\lpips_metrics_opencv_telea_50.csv
  feature_metrics_path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\feature_similarity_opencv_telea_50.csv

Model: lama
  classical_metrics_path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\classical_metrics_lama_50.csv
  lpips_metrics_path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\lpips_metrics_lama_50.csv
  feature_metrics_path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\feature_similarity_lama_50.csv

Model: stable_diffusion_inpainting
  classical_metrics_path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\classical_metrics_stable_diffusion_50.csv
  lpips_metrics_path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\me

In [4]:
classical_metrics = {}
lpips_metrics = {}
feature_metrics = {}

for model_name, model_info in model_registry.items():
    classical_metrics[model_name] = pd.read_csv(model_info["classical_metrics_path"])
    lpips_metrics[model_name] = pd.read_csv(model_info["lpips_metrics_path"])
    feature_metrics[model_name] = pd.read_csv(model_info["feature_metrics_path"])

    print(f"\n{model_name}")
    print("  classical:", classical_metrics[model_name].shape)
    print("  lpips:", lpips_metrics[model_name].shape)
    print("  feature:", feature_metrics[model_name].shape)

old_comparison_df = pd.read_csv(old_comparison_path)
old_win_rates_df = pd.read_csv(old_win_rates_path)

print("\nOld unified comparison:", old_comparison_df.shape)
print("Old win rates:", old_win_rates_df.shape)


opencv_telea
  classical: (900, 30)
  lpips: (700, 24)
  feature: (700, 28)

lama
  classical: (900, 30)
  lpips: (700, 24)
  feature: (700, 28)

stable_diffusion_inpainting
  classical: (900, 30)
  lpips: (700, 24)
  feature: (700, 28)

Old unified comparison: (200, 40)
Old win rates: (20, 5)


In [5]:
expected_classical_rows = 900
expected_feature_like_rows = 700

expected_classical_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "masked_region": 200,
    "mask_bbox_crop": 200,
}

expected_feature_like_region_counts = {
    "full_image": 250,
    "content_region": 250,
    "mask_bbox_crop": 200,
}

for model_name in model_registry:
    classical_df = classical_metrics[model_name]
    lpips_df = lpips_metrics[model_name]
    feature_df = feature_metrics[model_name]

    if len(classical_df) != expected_classical_rows:
        raise ValueError(
            f"{model_name}: expected {expected_classical_rows} classical rows, "
            f"found {len(classical_df)}."
        )

    if len(lpips_df) != expected_feature_like_rows:
        raise ValueError(
            f"{model_name}: expected {expected_feature_like_rows} LPIPS rows, "
            f"found {len(lpips_df)}."
        )

    if len(feature_df) != expected_feature_like_rows:
        raise ValueError(
            f"{model_name}: expected {expected_feature_like_rows} feature rows, "
            f"found {len(feature_df)}."
        )

    for metric_name, metric_df in {
        "classical": classical_df,
        "lpips": lpips_df,
        "feature": feature_df,
    }.items():
        if "status" in metric_df.columns and (metric_df["status"] != "ok").any():
            display(metric_df[metric_df["status"] != "ok"])
            raise ValueError(f"{model_name}: {metric_name} metrics contain non-ok rows.")

    actual_classical_regions = classical_df["evaluation_region"].value_counts().to_dict()

    for region, expected_count in expected_classical_region_counts.items():
        actual_count = actual_classical_regions.get(region, 0)

        if actual_count != expected_count:
            raise ValueError(
                f"{model_name}: classical region {region!r}: "
                f"expected {expected_count}, found {actual_count}."
            )

    for metric_name, metric_df in {
        "lpips": lpips_df,
        "feature": feature_df,
    }.items():
        actual_regions = metric_df["evaluation_region"].value_counts().to_dict()

        for region, expected_count in expected_feature_like_region_counts.items():
            actual_count = actual_regions.get(region, 0)

            if actual_count != expected_count:
                raise ValueError(
                    f"{model_name}: {metric_name} region {region!r}: "
                    f"expected {expected_count}, found {actual_count}."
                )

if len(old_comparison_df) != 200:
    raise ValueError(
        f"Expected 200 old comparison rows, found {len(old_comparison_df)}."
    )

print("Input metric validation gates passed.")

Input metric validation gates passed.


In [6]:
def find_single_column(df: pd.DataFrame, required_terms: list[str]) -> str:
    matches = []

    for column in df.columns:
        column_lower = column.lower()

        if all(term.lower() in column_lower for term in required_terms):
            matches.append(column)

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one column matching terms {required_terms}, "
            f"found {len(matches)}: {matches}"
        )

    return matches[0]


metric_column_registry = {}

for model_name in model_registry:
    classical_df = classical_metrics[model_name]
    lpips_df = lpips_metrics[model_name]
    feature_df = feature_metrics[model_name]

    metric_column_registry[model_name] = {
        "classical": {
            "mse_improvement": find_single_column(classical_df, ["mse", "improvement"]),
            "psnr_improvement": find_single_column(classical_df, ["psnr", "improvement"]),
            "ssim_improvement": find_single_column(classical_df, ["ssim", "improvement"]),
        },
        "lpips": {
            "lpips_improvement": find_single_column(lpips_df, ["lpips", "improvement"]),
        },
        "feature": {
            "clip_improvement": find_single_column(feature_df, ["clip", "improvement"]),
            "dinov2_improvement": find_single_column(feature_df, ["dinov2", "improvement"]),
        },
    }

print("Detected metric columns:")

for model_name, families in metric_column_registry.items():
    print("\n", model_name)

    for family_name, columns in families.items():
        print(" ", family_name)

        for metric_name, column_name in columns.items():
            print(f"    {metric_name}: {column_name}")

Detected metric columns:

 opencv_telea
  classical
    mse_improvement: mse_improvement
    psnr_improvement: psnr_improvement
    ssim_improvement: ssim_improvement
  lpips
    lpips_improvement: lpips_improvement
  feature
    clip_improvement: clip_similarity_improvement
    dinov2_improvement: dinov2_similarity_improvement

 lama
  classical
    mse_improvement: mse_improvement
    psnr_improvement: psnr_improvement
    ssim_improvement: ssim_improvement
  lpips
    lpips_improvement: lpips_improvement
  feature
    clip_improvement: clip_similarity_improvement
    dinov2_improvement: dinov2_similarity_improvement

 stable_diffusion_inpainting
  classical
    mse_improvement: mse_improvement
    psnr_improvement: psnr_improvement
    ssim_improvement: ssim_improvement
  lpips
    lpips_improvement: lpips_improvement
  feature
    clip_improvement: clip_similarity_improvement
    dinov2_improvement: dinov2_similarity_improvement


In [7]:
ssim_issue_rows = []

for model_name in model_registry:
    classical_df = classical_metrics[model_name].copy()
    ssim_column = metric_column_registry[model_name]["classical"]["ssim_improvement"]

    masked_ssim_df = classical_df[
        classical_df["evaluation_region"] == "masked_region"
    ].copy()

    bbox_ssim_df = classical_df[
        classical_df["evaluation_region"] == "mask_bbox_crop"
    ].copy()

    masked_valid_count = masked_ssim_df[ssim_column].notna().sum()
    bbox_valid_count = bbox_ssim_df[ssim_column].notna().sum()

    ssim_issue_rows.append(
        {
            "model_name": model_name,
            "masked_region_rows": len(masked_ssim_df),
            "masked_region_valid_ssim_rows": int(masked_valid_count),
            "masked_region_invalid_ssim_rows": int(len(masked_ssim_df) - masked_valid_count),
            "mask_bbox_crop_rows": len(bbox_ssim_df),
            "mask_bbox_crop_valid_ssim_rows": int(bbox_valid_count),
            "mask_bbox_crop_invalid_ssim_rows": int(len(bbox_ssim_df) - bbox_valid_count),
        }
    )

ssim_issue_df = pd.DataFrame(ssim_issue_rows)

display(ssim_issue_df)

if (ssim_issue_df["masked_region_valid_ssim_rows"] > 0).any():
    print(
        "Some sparse masked-region SSIM values are valid. "
        "Still using mask_bbox_crop for final policy because it is image-like."
    )
else:
    print("Sparse masked-region SSIM has no valid rows across models.")

if (ssim_issue_df["mask_bbox_crop_valid_ssim_rows"] == 0).any():
    raise ValueError(
        "At least one model has no valid mask_bbox_crop SSIM values. "
        "Cannot use refined SSIM policy."
    )

print("SSIM issue confirmation complete.")

,model_name,masked_region_rows,masked_region_valid_ssim_rows,masked_region_invalid_ssim_rows,mask_bbox_crop_rows,mask_bbox_crop_valid_ssim_rows,mask_bbox_crop_invalid_ssim_rows
0,opencv_telea,200,0,200,200,200,0
1,lama,200,0,200,200,200,0
2,stable_diffusion_inpainting,200,0,200,200,200,0


Sparse masked-region SSIM has no valid rows across models.
SSIM issue confirmation complete.


## Refined final local metric policy

The initial comparison used sparse `masked_region` values for MSE, PSNR, and SSIM.

The validation above confirms that sparse masked-region SSIM is not suitable for the final local comparison. SSIM requires an image-like region with local spatial context, while the sparse masked region does not provide a stable structural neighborhood.

SSIM is therefore retained, but moved to the `mask_bbox_crop` region.

Final local comparison policy:

| Metric | Region | Reason |
|---|---|---|
| MSE improvement | `masked_region` | pixel-error metric directly over damaged pixels |
| PSNR improvement | `masked_region` | pixel-error metric directly over damaged pixels |
| SSIM improvement | `mask_bbox_crop` | structural metric requiring image-like context |
| LPIPS improvement | `mask_bbox_crop` | perceptual metric requiring image-like input |
| CLIP similarity improvement | `mask_bbox_crop` | feature metric requiring image-like input |
| DINOv2 similarity improvement | `mask_bbox_crop` | feature metric requiring image-like input |

In [8]:
refined_metric_policy = {
    "mse_improvement": {
        "family": "classical",
        "region": "masked_region",
        "higher_is_better": True,
        "description": "MSE improvement on sparse masked region",
    },
    "psnr_improvement": {
        "family": "classical",
        "region": "masked_region",
        "higher_is_better": True,
        "description": "PSNR improvement on sparse masked region",
    },
    "ssim_improvement": {
        "family": "classical",
        "region": "mask_bbox_crop",
        "higher_is_better": True,
        "description": "SSIM improvement on mask-bounding-box crop",
    },
    "lpips_improvement": {
        "family": "lpips",
        "region": "mask_bbox_crop",
        "higher_is_better": True,
        "description": "LPIPS improvement on mask-bounding-box crop",
    },
    "clip_improvement": {
        "family": "feature",
        "region": "mask_bbox_crop",
        "higher_is_better": True,
        "description": "CLIP similarity improvement on mask-bounding-box crop",
    },
    "dinov2_improvement": {
        "family": "feature",
        "region": "mask_bbox_crop",
        "higher_is_better": True,
        "description": "DINOv2 similarity improvement on mask-bounding-box crop",
    },
}

model_names = list(model_registry.keys())

print("Refined metric policy:")

for metric_name, metric_info in refined_metric_policy.items():
    print(
        f"- {metric_name}: {metric_info['family']} | "
        f"{metric_info['region']} | {metric_info['description']}"
    )

Refined metric policy:
- mse_improvement: classical | masked_region | MSE improvement on sparse masked region
- psnr_improvement: classical | masked_region | PSNR improvement on sparse masked region
- ssim_improvement: classical | mask_bbox_crop | SSIM improvement on mask-bounding-box crop
- lpips_improvement: lpips | mask_bbox_crop | LPIPS improvement on mask-bounding-box crop
- clip_improvement: feature | mask_bbox_crop | CLIP similarity improvement on mask-bounding-box crop
- dinov2_improvement: feature | mask_bbox_crop | DINOv2 similarity improvement on mask-bounding-box crop


In [9]:
base_columns = [
    "case_id",
    "painting_id",
    "mask_id",
    "mask_type",
    "category",
    "title",
]


def get_metric_source_df(model_name: str, family: str) -> pd.DataFrame:
    if family == "classical":
        return classical_metrics[model_name]

    if family == "lpips":
        return lpips_metrics[model_name]

    if family == "feature":
        return feature_metrics[model_name]

    raise ValueError(f"Unknown metric family: {family}")


def get_metric_column_name(model_name: str, family: str, metric_name: str) -> str:
    if family == "classical":
        return metric_column_registry[model_name]["classical"][metric_name]

    if family == "lpips":
        return metric_column_registry[model_name]["lpips"][metric_name]

    if family == "feature":
        return metric_column_registry[model_name]["feature"][metric_name]

    raise ValueError(f"Unknown metric family: {family}")


def extract_refined_local_model_metrics(model_name: str) -> pd.DataFrame:
    model_metric_frames = []
    base_df = None

    for metric_name, metric_info in refined_metric_policy.items():
        family = metric_info["family"]
        region = metric_info["region"]

        source_df = get_metric_source_df(model_name, family).copy()
        metric_column = get_metric_column_name(model_name, family, metric_name)

        region_df = source_df[
            source_df["evaluation_region"] == region
        ].copy()

        if len(region_df) != 200:
            raise ValueError(
                f"{model_name} | {metric_name} | {region}: "
                f"expected 200 rows, found {len(region_df)}."
            )

        missing_base_columns = [
            column for column in base_columns
            if column not in region_df.columns
        ]

        if missing_base_columns:
            raise ValueError(
                f"{model_name} | {metric_name}: missing base columns: "
                f"{missing_base_columns}"
            )

        metric_values_df = region_df[
            base_columns + [metric_column]
        ].copy()

        if metric_values_df[metric_column].isna().any():
            invalid_rows = metric_values_df[metric_values_df[metric_column].isna()]
            display(invalid_rows.head())
            raise ValueError(
                f"{model_name} | {metric_name} | {region}: contains NaN values."
            )

        metric_values_df = metric_values_df.rename(
            columns={
                metric_column: f"{model_name}_{metric_name}",
            }
        )

        if base_df is None:
            base_df = metric_values_df[base_columns].copy()
            if "title" in base_df.columns:
                base_df["title"] = base_df["title"].fillna("").astype(str)

        model_metric_frames.append(
            metric_values_df[
                ["case_id", f"{model_name}_{metric_name}"]
            ].copy()
        )

    model_local_df = base_df.copy()

    for metric_df in model_metric_frames:
        model_local_df = model_local_df.merge(
            metric_df,
            on="case_id",
            how="inner",
            validate="one_to_one",
        )

    if len(model_local_df) != 200:
        raise ValueError(
            f"{model_name}: expected 200 refined local rows, "
            f"found {len(model_local_df)}."
        )

    return model_local_df


refined_model_local_tables = {
    model_name: extract_refined_local_model_metrics(model_name)
    for model_name in model_names
}

for model_name, model_local_df in refined_model_local_tables.items():
    print(model_name, model_local_df.shape)
    display(model_local_df.head(2))

opencv_telea (200, 12)


,case_id,painting_id,mask_id,mask_type,category,title,opencv_telea_mse_improvement,opencv_telea_psnr_improvement,opencv_telea_ssim_improvement,opencv_telea_lpips_improvement,opencv_telea_clip_improvement,opencv_telea_dinov2_improvement
0,p001_loss_large,p001,p001_loss_large,loss_large,portrait_figure,Juan de Pareja,48015.951050,16.981299,0.307164,0.318226,-0.002892,0.041827
1,p001_loss_small,p001,p001_loss_small,loss_small,portrait_figure,Juan de Pareja,46962.810768,31.576516,0.079075,0.333183,0.088604,0.048946


lama (200, 12)


,case_id,painting_id,mask_id,mask_type,category,title,lama_mse_improvement,lama_psnr_improvement,lama_ssim_improvement,lama_lpips_improvement,lama_clip_improvement,lama_dinov2_improvement
0,p001_loss_large,p001,p001_loss_large,loss_large,portrait_figure,,48703.631836,22.215740,0.319821,0.444638,0.166787,0.253627
1,p001_loss_small,p001,p001_loss_small,loss_small,portrait_figure,,46980.140714,34.856852,0.078012,0.338072,0.089109,0.050690


stable_diffusion_inpainting (200, 12)


,case_id,painting_id,mask_id,mask_type,category,title,stable_diffusion_inpainting_mse_improvement,stable_diffusion_inpainting_psnr_improvement,stable_diffusion_inpainting_ssim_improvement,stable_diffusion_inpainting_lpips_improvement,stable_diffusion_inpainting_clip_improvement,stable_diffusion_inpainting_dinov2_improvement
0,p001_loss_large,p001,p001_loss_large,loss_large,portrait_figure,Juan de Pareja,48133.030640,17.532740,0.118627,0.391132,0.119137,0.248403
1,p001_loss_small,p001,p001_loss_small,loss_small,portrait_figure,Juan de Pareja,46912.756538,27.543226,-0.077822,0.306002,0.071142,0.037632


In [10]:
identity_columns = [
    "case_id",
    "painting_id",
    "mask_id",
    "mask_type",
    "category",
]

metadata_columns = [
    "title",
]

refined_unified_df = refined_model_local_tables[model_names[0]].copy()

if "title" in refined_unified_df.columns:
    refined_unified_df["title"] = refined_unified_df["title"].fillna("").astype(str)

for model_name in model_names[1:]:
    model_table = refined_model_local_tables[model_name].copy()

    if "title" in model_table.columns:
        model_table["title"] = model_table["title"].fillna("").astype(str)

    model_metric_columns = [
        column
        for column in model_table.columns
        if column not in identity_columns + metadata_columns
    ]

    refined_unified_df = refined_unified_df.merge(
        model_table[identity_columns + model_metric_columns],
        on=identity_columns,
        how="inner",
        validate="one_to_one",
    )

if len(refined_unified_df) != 200:
    raise ValueError(
        f"Expected 200 refined unified rows, found {len(refined_unified_df)}."
    )

if refined_unified_df["case_id"].duplicated().any():
    duplicated_cases = refined_unified_df[
        refined_unified_df["case_id"].duplicated(keep=False)
    ]
    display(duplicated_cases)
    raise ValueError("Refined unified comparison contains duplicated case IDs.")

refined_unified_df.to_csv(output_paths["refined_unified"], index=False)

print("Refined unified comparison rows:", len(refined_unified_df))
print("Saved refined unified comparison:", output_paths["refined_unified"])

display(refined_unified_df.head())

Refined unified comparison rows: 200
Saved refined unified comparison: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_unified_refined_opencv_lama_stable_diffusion_50.csv


,case_id,painting_id,mask_id,mask_type,category,title,opencv_telea_mse_improvement,opencv_telea_psnr_improvement,opencv_telea_ssim_improvement,opencv_telea_lpips_improvement,...,lama_ssim_improvement,lama_lpips_improvement,lama_clip_improvement,lama_dinov2_improvement,stable_diffusion_inpainting_mse_improvement,stable_diffusion_inpainting_psnr_improvement,stable_diffusion_inpainting_ssim_improvement,stable_diffusion_inpainting_lpips_improvement,stable_diffusion_inpainting_clip_improvement,stable_diffusion_inpainting_dinov2_improvement
0,p001_loss_large,p001,p001_loss_large,loss_large,portrait_figure,Juan de Pareja,48015.951050,16.981299,0.307164,0.318226,...,0.319821,0.444638,0.166787,0.253627,48133.030640,17.532740,0.118627,0.391132,0.119137,0.248403
1,p001_loss_small,p001,p001_loss_small,loss_small,portrait_figure,Juan de Pareja,46962.810768,31.576516,0.079075,0.333183,...,0.078012,0.338072,0.089109,0.050690,46912.756538,27.543226,-0.077822,0.306002,0.071142,0.037632
2,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,portrait_figure,Juan de Pareja,50520.931183,23.633919,0.085103,0.385844,...,0.083534,0.391770,0.102389,0.197233,49826.728882,17.444243,-0.093749,0.244464,0.060504,0.167937
3,p001_scratch_thin,p001,p001_scratch_thin,scratch_thin,portrait_figure,Juan de Pareja,48335.436569,24.474660,0.047276,0.419886,...,0.047212,0.420297,0.141728,0.074596,42301.111328,8.929056,-0.153775,0.161720,0.043441,-0.005591
4,p002_loss_large,p002,p002_loss_large,loss_large,portrait_figure,Madame X (Madame Pierre Gautreau),50914.557739,22.108130,0.375796,0.387116,...,0.405204,0.448236,0.245552,0.172309,50524.246399,18.609613,0.212932,0.407476,0.222473,0.119677


In [11]:
def metric_column_for_model(model_name: str, metric_name: str) -> str:
    return f"{model_name}_{metric_name}"


def determine_metric_winner(row: pd.Series, metric_name: str) -> str:
    values = {}

    for model_name in model_names:
        column_name = metric_column_for_model(model_name, metric_name)

        if column_name not in row.index:
            raise ValueError(f"Missing metric column: {column_name}")

        value = row[column_name]

        if pd.notna(value):
            values[model_name] = value

    if not values:
        return "no_valid_values"

    max_value = max(values.values())

    winners = [
        model_name
        for model_name, value in values.items()
        if value == max_value
    ]

    if len(winners) == len(values):
        return "tie_all"

    if len(winners) > 1:
        return "tie_" + "_".join(sorted(winners))

    return winners[0]


def count_model_wins(row: pd.Series) -> pd.Series:
    win_counts = {
        f"{model_name}_metric_wins": 0
        for model_name in model_names
    }

    tie_count = 0

    for metric_name in refined_metric_policy:
        winner = row[f"winner_{metric_name}"]

        if winner in model_names:
            win_counts[f"{winner}_metric_wins"] += 1
        else:
            tie_count += 1

    win_counts["metric_ties"] = tie_count

    max_wins = max(
        win_counts[f"{model_name}_metric_wins"]
        for model_name in model_names
    )

    top_models = [
        model_name
        for model_name in model_names
        if win_counts[f"{model_name}_metric_wins"] == max_wins
    ]

    if len(top_models) == 1 and max_wins > 0:
        win_counts["overall_metric_vote"] = top_models[0]
    elif max_wins == 0:
        win_counts["overall_metric_vote"] = "all_tie_or_no_winner"
    else:
        win_counts["overall_metric_vote"] = "tie_" + "_".join(sorted(top_models))

    return pd.Series(win_counts)


print("Refined winner helpers ready.")

Refined winner helpers ready.


In [12]:
refined_comparison_df = refined_unified_df.copy()

for metric_name in refined_metric_policy:
    for model_name in model_names:
        column_name = metric_column_for_model(model_name, metric_name)

        if column_name not in refined_comparison_df.columns:
            raise ValueError(f"Missing refined comparison metric column: {column_name}")

    refined_comparison_df[f"winner_{metric_name}"] = refined_comparison_df.apply(
        lambda row: determine_metric_winner(row, metric_name),
        axis=1,
    )

refined_win_count_df = refined_comparison_df.apply(count_model_wins, axis=1)

refined_comparison_df = pd.concat(
    [
        refined_comparison_df,
        refined_win_count_df,
    ],
    axis=1,
)

winner_columns = [
    f"winner_{metric_name}"
    for metric_name in refined_metric_policy
]

refined_comparison_df["mixed_metric_outcome"] = (
    refined_comparison_df[winner_columns].nunique(axis=1) > 1
)

refined_comparison_df["all_metrics_same_winner"] = (
    refined_comparison_df[winner_columns].nunique(axis=1) == 1
)

for model_name in model_names:
    refined_comparison_df[f"{model_name}_all_metrics_win"] = (
        refined_comparison_df[winner_columns].eq(model_name).all(axis=1)
    )

refined_comparison_df.to_csv(output_paths["refined_unified"], index=False)

print("Refined comparison rows:", len(refined_comparison_df))
print("Saved refined comparison:", output_paths["refined_unified"])

display(
    refined_comparison_df[
        [
            "case_id",
            "category",
            "mask_type",
            *winner_columns,
            *[f"{model_name}_metric_wins" for model_name in model_names],
            "metric_ties",
            "overall_metric_vote",
            "mixed_metric_outcome",
        ]
    ].head()
)

Refined comparison rows: 200
Saved refined comparison: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_unified_refined_opencv_lama_stable_diffusion_50.csv


,case_id,category,mask_type,winner_mse_improvement,winner_psnr_improvement,winner_ssim_improvement,winner_lpips_improvement,winner_clip_improvement,winner_dinov2_improvement,opencv_telea_metric_wins,lama_metric_wins,stable_diffusion_inpainting_metric_wins,metric_ties,overall_metric_vote,mixed_metric_outcome
0,p001_loss_large,portrait_figure,loss_large,lama,lama,lama,lama,lama,lama,0,6,0,0,lama,False
1,p001_loss_small,portrait_figure,loss_small,lama,lama,opencv_telea,lama,lama,lama,1,5,0,0,lama,True
2,p001_mixed_damage,portrait_figure,mixed_damage,opencv_telea,opencv_telea,opencv_telea,lama,lama,lama,3,3,0,0,tie_lama_opencv_telea,True
3,p001_scratch_thin,portrait_figure,scratch_thin,lama,lama,opencv_telea,lama,lama,lama,1,5,0,0,lama,True
4,p002_loss_large,portrait_figure,loss_large,lama,lama,lama,lama,lama,lama,0,6,0,0,lama,False


In [13]:
refined_winner_rows = []

for metric_name, metric_info in refined_metric_policy.items():
    winner_counts = refined_comparison_df[f"winner_{metric_name}"].value_counts(dropna=False)

    for winner_name, count in winner_counts.items():
        refined_winner_rows.append(
            {
                "metric_name": metric_name,
                "metric_description": metric_info["description"],
                "metric_region": metric_info["region"],
                "winner": winner_name,
                "cases": int(count),
                "case_fraction": count / len(refined_comparison_df),
            }
        )

overall_vote_counts = refined_comparison_df["overall_metric_vote"].value_counts(dropna=False)

for winner_name, count in overall_vote_counts.items():
    refined_winner_rows.append(
        {
            "metric_name": "overall_metric_vote",
            "metric_description": "Per-case majority vote across six refined local metrics",
            "metric_region": "mixed_refined_regions",
            "winner": winner_name,
            "cases": int(count),
            "case_fraction": count / len(refined_comparison_df),
        }
    )

refined_win_rates_df = pd.DataFrame(refined_winner_rows)

refined_win_rates_df.to_csv(
    output_paths["refined_win_rates"],
    index=False,
)

print("Refined win rates:")
display(refined_win_rates_df)

print("Saved refined win rates:", output_paths["refined_win_rates"])

Refined win rates:


,metric_name,metric_description,metric_region,winner,cases,case_fraction
0,mse_improvement,MSE improvement on sparse masked region,masked_region,lama,159,0.795
1,mse_improvement,MSE improvement on sparse masked region,masked_region,opencv_telea,40,0.200
2,mse_improvement,MSE improvement on sparse masked region,masked_region,stable_diffusion_inpainting,1,0.005
3,psnr_improvement,PSNR improvement on sparse masked region,masked_region,lama,159,0.795
4,psnr_improvement,PSNR improvement on sparse masked region,masked_region,opencv_telea,40,0.200
5,psnr_improvement,PSNR improvement on sparse masked region,masked_region,stable_diffusion_inpainting,1,0.005
6,ssim_improvement,SSIM improvement on mask-bounding-box crop,mask_bbox_crop,opencv_telea,113,0.565
7,ssim_improvement,SSIM improvement on mask-bounding-box crop,mask_bbox_crop,lama,87,0.435
8,lpips_improvement,LPIPS improvement on mask-bounding-box crop,mask_bbox_crop,lama,182,0.910
9,lpips_improvement,LPIPS improvement on mask-bounding-box crop,mask_bbox_crop,opencv_telea,12,0.060


Saved refined win rates: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_win_rates_refined_opencv_lama_stable_diffusion_50.csv


In [14]:
summary_agg = {
    "cases": ("case_id", "count"),
    "mean_metric_ties": ("metric_ties", "mean"),
}

for model_name in model_names:
    summary_agg[f"mean_{model_name}_metric_wins"] = (
        f"{model_name}_metric_wins",
        "mean",
    )

for metric_name in refined_metric_policy:
    for model_name in model_names:
        summary_agg[f"mean_{model_name}_{metric_name}"] = (
            f"{model_name}_{metric_name}",
            "mean",
        )

refined_summary_by_mask_type_df = (
    refined_comparison_df
    .groupby("mask_type", dropna=False)
    .agg(**summary_agg)
    .reset_index()
    .sort_values("mask_type")
)

refined_summary_by_category_df = (
    refined_comparison_df
    .groupby("category", dropna=False)
    .agg(**summary_agg)
    .reset_index()
    .sort_values("category")
)

refined_summary_by_mask_type_df.to_csv(
    output_paths["refined_summary_by_mask_type"],
    index=False,
)

refined_summary_by_category_df.to_csv(
    output_paths["refined_summary_by_category"],
    index=False,
)

print("Refined summary by mask type:")
display(refined_summary_by_mask_type_df)

print("\nRefined summary by category:")
display(refined_summary_by_category_df)

print("\nSaved refined summaries:")
print(output_paths["refined_summary_by_mask_type"])
print(output_paths["refined_summary_by_category"])

Refined summary by mask type:


,mask_type,cases,mean_metric_ties,mean_opencv_telea_metric_wins,mean_lama_metric_wins,mean_stable_diffusion_inpainting_metric_wins,mean_opencv_telea_mse_improvement,mean_lama_mse_improvement,mean_stable_diffusion_inpainting_mse_improvement,mean_opencv_telea_psnr_improvement,...,mean_stable_diffusion_inpainting_ssim_improvement,mean_opencv_telea_lpips_improvement,mean_lama_lpips_improvement,mean_stable_diffusion_inpainting_lpips_improvement,mean_opencv_telea_clip_improvement,mean_lama_clip_improvement,mean_stable_diffusion_inpainting_clip_improvement,mean_opencv_telea_dinov2_improvement,mean_lama_dinov2_improvement,mean_stable_diffusion_inpainting_dinov2_improvement
0,loss_large,50,0.0,0.80,4.82,0.38,23999.733308,24555.375813,23124.429134,12.615486,...,-0.099097,0.185879,0.307958,0.221407,0.026314,0.158192,0.106688,-0.122129,0.135501,0.039577
1,loss_small,50,0.0,0.82,5.18,0.00,25902.881564,26147.753708,25636.699247,17.393119,...,-0.263091,0.166526,0.182632,0.108700,0.098812,0.103704,0.073849,0.057932,0.078849,0.020351
2,mixed_damage,50,0.0,0.72,5.28,0.00,26381.819261,26669.089760,25735.033149,15.395991,...,-0.235954,0.264452,0.290510,0.188334,0.132395,0.149796,0.104924,0.172798,0.231918,0.110759
3,scratch_thin,50,0.0,2.80,3.20,0.00,28319.668946,28387.473466,25670.202737,20.491108,...,-0.274728,0.256587,0.258198,0.131230,0.101158,0.101232,0.048619,0.147648,0.146852,0.018038



Refined summary by category:


,category,cases,mean_metric_ties,mean_opencv_telea_metric_wins,mean_lama_metric_wins,mean_stable_diffusion_inpainting_metric_wins,mean_opencv_telea_mse_improvement,mean_lama_mse_improvement,mean_stable_diffusion_inpainting_mse_improvement,mean_opencv_telea_psnr_improvement,...,mean_stable_diffusion_inpainting_ssim_improvement,mean_opencv_telea_lpips_improvement,mean_lama_lpips_improvement,mean_stable_diffusion_inpainting_lpips_improvement,mean_opencv_telea_clip_improvement,mean_lama_clip_improvement,mean_stable_diffusion_inpainting_clip_improvement,mean_opencv_telea_dinov2_improvement,mean_lama_dinov2_improvement,mean_stable_diffusion_inpainting_dinov2_improvement
0,abstraction_surrealism,40,0.0,1.250,4.575,0.175,18899.959899,19484.237353,17876.773131,12.663342,...,-0.203084,0.139129,0.166960,0.105644,0.065088,0.087756,0.063532,0.026611,0.091159,0.014449
1,architecture_structured,40,0.0,1.400,4.525,0.075,25406.392617,25722.202983,24537.514644,17.020632,...,-0.213232,0.238820,0.287507,0.188694,0.111361,0.151707,0.095640,0.033980,0.103146,0.008588
2,high_texture_brushwork,40,0.0,1.375,4.500,0.125,26778.382584,26948.319574,25769.689114,17.441999,...,-0.254822,0.236506,0.276681,0.173777,0.091352,0.140614,0.099730,0.093131,0.175133,0.103444
3,landscape_natural,40,0.0,1.550,4.425,0.025,21573.904087,21766.124514,20384.583746,15.243420,...,-0.275494,0.207191,0.263705,0.136462,0.090349,0.138341,0.069364,0.087183,0.229342,0.045523
4,portrait_figure,40,0.0,0.850,5.075,0.075,38096.489662,38278.731508,36639.394696,20.000236,...,-0.144454,0.270158,0.304269,0.207513,0.090200,0.122736,0.089334,0.079405,0.142620,0.063903



Saved refined summaries:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_summary_by_mask_type_refined_opencv_lama_stable_diffusion_50.csv
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_summary_by_category_refined_opencv_lama_stable_diffusion_50.csv


In [15]:
refined_metric_disagreement_cases_df = refined_comparison_df[
    refined_comparison_df["mixed_metric_outcome"]
].copy()

refined_metric_disagreement_cases_df["num_unique_metric_winners"] = (
    refined_metric_disagreement_cases_df[winner_columns].nunique(axis=1)
)

refined_metric_disagreement_cases_df["max_model_metric_wins"] = refined_metric_disagreement_cases_df[
    [f"{model_name}_metric_wins" for model_name in model_names]
].max(axis=1)

refined_metric_disagreement_cases_df["disagreement_strength"] = (
    refined_metric_disagreement_cases_df["num_unique_metric_winners"]
    + refined_metric_disagreement_cases_df["metric_ties"]
)

refined_metric_disagreement_cases_df = (
    refined_metric_disagreement_cases_df
    .sort_values(
        [
            "disagreement_strength",
            "num_unique_metric_winners",
            "metric_ties",
        ],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

refined_metric_disagreement_cases_df.to_csv(
    output_paths["refined_metric_disagreement_cases"],
    index=False,
)

print("Refined metric-disagreement cases:", len(refined_metric_disagreement_cases_df))
print("Saved:", output_paths["refined_metric_disagreement_cases"])

display(
    refined_metric_disagreement_cases_df[
        [
            "case_id",
            "category",
            "mask_type",
            "overall_metric_vote",
            "num_unique_metric_winners",
            "disagreement_strength",
            *winner_columns,
            *[f"{model_name}_metric_wins" for model_name in model_names],
            "metric_ties",
        ]
    ].head(30)
)

Refined metric-disagreement cases: 124
Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_metric_disagreement_cases_refined_opencv_lama_stable_diffusion_50.csv


,case_id,category,mask_type,overall_metric_vote,num_unique_metric_winners,disagreement_strength,winner_mse_improvement,winner_psnr_improvement,winner_ssim_improvement,winner_lpips_improvement,winner_clip_improvement,winner_dinov2_improvement,opencv_telea_metric_wins,lama_metric_wins,stable_diffusion_inpainting_metric_wins,metric_ties
0,p009_loss_large,portrait_figure,loss_large,lama,3,3,lama,lama,opencv_telea,lama,stable_diffusion_inpainting,stable_diffusion_inpainting,1,3,2,0
1,p024_loss_large,architecture_structured,loss_large,lama,3,3,lama,lama,opencv_telea,stable_diffusion_inpainting,lama,lama,1,4,1,0
2,p037_loss_large,abstraction_surrealism,loss_large,stable_diffusion_inpainting,3,3,stable_diffusion_inpainting,stable_diffusion_inpainting,opencv_telea,stable_diffusion_inpainting,stable_diffusion_inpainting,lama,1,1,4,0
3,p040_loss_large,abstraction_surrealism,loss_large,opencv_telea,3,3,opencv_telea,opencv_telea,opencv_telea,lama,stable_diffusion_inpainting,lama,3,2,1,0
4,p046_loss_large,high_texture_brushwork,loss_large,lama,3,3,lama,lama,opencv_telea,lama,lama,stable_diffusion_inpainting,1,4,1,0
5,p048_loss_large,high_texture_brushwork,loss_large,lama,3,3,lama,lama,opencv_telea,lama,stable_diffusion_inpainting,lama,1,4,1,0
6,p050_loss_large,high_texture_brushwork,loss_large,lama,3,3,lama,lama,opencv_telea,lama,lama,stable_diffusion_inpainting,1,4,1,0
7,p001_loss_small,portrait_figure,loss_small,lama,2,2,lama,lama,opencv_telea,lama,lama,lama,1,5,0,0
8,p001_mixed_damage,portrait_figure,mixed_damage,tie_lama_opencv_telea,2,2,opencv_telea,opencv_telea,opencv_telea,lama,lama,lama,3,3,0,0
9,p001_scratch_thin,portrait_figure,scratch_thin,lama,2,2,lama,lama,opencv_telea,lama,lama,lama,1,5,0,0


In [16]:
old_vote_columns = [
    "case_id",
    "category",
    "mask_type",
    "overall_metric_vote",
]

missing_old_vote_columns = [
    column for column in old_vote_columns
    if column not in old_comparison_df.columns
]

if missing_old_vote_columns:
    raise ValueError(
        f"Old comparison missing vote columns: {missing_old_vote_columns}"
    )

old_votes_df = old_comparison_df[old_vote_columns].copy()
old_votes_df = old_votes_df.rename(
    columns={
        "overall_metric_vote": "old_overall_metric_vote",
    }
)

refined_votes_df = refined_comparison_df[
    [
        "case_id",
        "overall_metric_vote",
        *[f"{model_name}_metric_wins" for model_name in model_names],
        "metric_ties",
    ]
].copy()

refined_votes_df = refined_votes_df.rename(
    columns={
        "overall_metric_vote": "refined_overall_metric_vote",
        **{
            f"{model_name}_metric_wins": f"refined_{model_name}_metric_wins"
            for model_name in model_names
        },
        "metric_ties": "refined_metric_ties",
    }
)

old_vs_refined_df = old_votes_df.merge(
    refined_votes_df,
    on="case_id",
    how="inner",
    validate="one_to_one",
)

if len(old_vs_refined_df) != 200:
    raise ValueError(
        f"Expected 200 old-vs-refined rows, found {len(old_vs_refined_df)}."
    )

old_vs_refined_df["overall_vote_changed"] = (
    old_vs_refined_df["old_overall_metric_vote"]
    != old_vs_refined_df["refined_overall_metric_vote"]
)

old_vs_refined_df.to_csv(output_paths["old_vs_refined"], index=False)

vote_change_summary_df = (
    old_vs_refined_df
    .groupby(
        [
            "old_overall_metric_vote",
            "refined_overall_metric_vote",
        ],
        dropna=False,
    )
    .agg(cases=("case_id", "count"))
    .reset_index()
    .sort_values("cases", ascending=False)
)

print("Overall vote changes:", int(old_vs_refined_df["overall_vote_changed"].sum()))
print("Saved old-vs-refined table:", output_paths["old_vs_refined"])

display(vote_change_summary_df)

print("\nChanged cases:")
display(
    old_vs_refined_df[
        old_vs_refined_df["overall_vote_changed"]
    ][
        [
            "case_id",
            "category",
            "mask_type",
            "old_overall_metric_vote",
            "refined_overall_metric_vote",
        ]
    ].head(50)
)

Overall vote changes: 24
Saved old-vs-refined table: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_old_vs_refined_metric_policy_50.csv


,old_overall_metric_vote,refined_overall_metric_vote,cases
0,lama,lama,155
1,lama,tie_lama_opencv_telea,22
2,opencv_telea,opencv_telea,20
3,opencv_telea,tie_lama_opencv_telea,1
4,stable_diffusion_inpainting,stable_diffusion_inpainting,1
5,tie_lama_opencv_telea,opencv_telea,1



Changed cases:


,case_id,category,mask_type,old_overall_metric_vote,refined_overall_metric_vote
2,p001_mixed_damage,portrait_figure,mixed_damage,lama,tie_lama_opencv_telea
15,p004_scratch_thin,portrait_figure,scratch_thin,lama,tie_lama_opencv_telea
36,p010_loss_large,portrait_figure,loss_large,lama,tie_lama_opencv_telea
37,p010_loss_small,portrait_figure,loss_small,lama,tie_lama_opencv_telea
39,p010_scratch_thin,portrait_figure,scratch_thin,lama,tie_lama_opencv_telea
56,p015_loss_large,landscape_natural,loss_large,lama,tie_lama_opencv_telea
57,p015_loss_small,landscape_natural,loss_small,lama,tie_lama_opencv_telea
62,p016_mixed_damage,landscape_natural,mixed_damage,lama,tie_lama_opencv_telea
63,p016_scratch_thin,landscape_natural,scratch_thin,lama,tie_lama_opencv_telea
64,p017_loss_large,landscape_natural,loss_large,lama,tie_lama_opencv_telea


In [17]:
previous_visual_cases_path = metrics_dir / "comparison_visual_cases_opencv_lama_stable_diffusion_50.csv"

if not previous_visual_cases_path.exists():
    raise FileNotFoundError(
        f"Missing previous visual comparison cases file: {previous_visual_cases_path}"
    )

previous_visual_cases_df = pd.read_csv(previous_visual_cases_path)

required_previous_visual_columns = [
    "case_id",
    "comparison_figure_path",
]

missing_previous_visual_columns = [
    column for column in required_previous_visual_columns
    if column not in previous_visual_cases_df.columns
]

if missing_previous_visual_columns:
    raise ValueError(
        f"Previous visual cases file missing columns: {missing_previous_visual_columns}"
    )

print("Previous visual cases:", previous_visual_cases_df.shape)
display(previous_visual_cases_df.head())

Previous visual cases: (43, 57)


,case_id,painting_id,mask_id,mask_type,category,title,opencv_telea_mse_improvement,opencv_telea_psnr_improvement,opencv_telea_ssim_improvement,opencv_telea_lpips_improvement,...,opencv_telea_mask_path,lama_clean_path,lama_damaged_path,lama_restored_path,lama_mask_path,stable_diffusion_inpainting_clean_path,stable_diffusion_inpainting_damaged_path,stable_diffusion_inpainting_restored_path,stable_diffusion_inpainting_mask_path,comparison_figure_path
0,p031_loss_large,p031,p031_loss_large,loss_large,abstraction_surrealism,Improvisation No. 30 (Cannons),16148.701782,10.541362,NaN,0.112901,...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p031_loss_large_r...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/stable_diffusion_inpai...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...
1,p037_loss_large,p037,p037_loss_large,loss_large,abstraction_surrealism,In the Sea,32042.753418,13.630637,NaN,0.240938,...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p037_loss_large_r...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/stable_diffusion_inpai...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...
2,p040_loss_large,p040,p040_loss_large,loss_large,abstraction_surrealism,Houses at Murnau,29661.279053,11.368210,NaN,0.146600,...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p040_loss_large_r...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/stable_diffusion_inpai...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...
3,p031_loss_small,p031,p031_loss_small,loss_small,abstraction_surrealism,Improvisation No. 30 (Cannons),17825.910400,12.590339,NaN,0.072478,...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p031_loss_small_r...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/stable_diffusion_inpai...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...
4,p032_loss_small,p032,p032_loss_small,loss_small,abstraction_surrealism,Composition (No. 1) Gray-Red,5503.778564,4.674150,NaN,0.090158,...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/lama/p032_loss_small_r...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,data/processed/restored/stable_diffusion_inpai...,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...


In [18]:
refined_visual_selection_frames = []

# Strongest refined majority-vote cases by model.
for model_name in model_names:
    model_majority_df = refined_comparison_df[
        refined_comparison_df["overall_metric_vote"] == model_name
    ].copy()

    if not model_majority_df.empty:
        model_majority_df["refined_selection_reason"] = f"refined_majority_vote_{model_name}"
        refined_visual_selection_frames.append(
            model_majority_df
            .sort_values(f"{model_name}_metric_wins", ascending=False)
            .head(5)
        )

# Strongest disagreement cases.
if not refined_metric_disagreement_cases_df.empty:
    disagreement_visual_df = refined_metric_disagreement_cases_df.head(12).copy()
    disagreement_visual_df["refined_selection_reason"] = "refined_metric_disagreement"
    refined_visual_selection_frames.append(disagreement_visual_df)

# Old-vs-refined changed cases.
changed_vote_cases_df = old_vs_refined_df[
    old_vs_refined_df["overall_vote_changed"]
].copy()

if not changed_vote_cases_df.empty:
    changed_vote_case_ids = changed_vote_cases_df["case_id"].tolist()

    changed_vote_visual_df = refined_comparison_df[
        refined_comparison_df["case_id"].isin(changed_vote_case_ids)
    ].copy()

    changed_vote_visual_df["refined_selection_reason"] = "old_vs_refined_vote_changed"
    refined_visual_selection_frames.append(changed_vote_visual_df)

# Category/mask representatives under refined policy.
category_mask_refined_visual_df = refined_comparison_df.copy()
category_mask_refined_visual_df["max_model_metric_wins"] = category_mask_refined_visual_df[
    [f"{model_name}_metric_wins" for model_name in model_names]
].max(axis=1)

category_mask_refined_visual_df = (
    category_mask_refined_visual_df
    .sort_values("max_model_metric_wins", ascending=False)
    .groupby(["category", "mask_type"], dropna=False)
    .head(1)
    .copy()
)

category_mask_refined_visual_df["refined_selection_reason"] = "refined_category_mask_representative"
refined_visual_selection_frames.append(category_mask_refined_visual_df)

refined_visual_cases_raw_df = pd.concat(
    refined_visual_selection_frames,
    ignore_index=True,
)

refined_selection_reason_df = (
    refined_visual_cases_raw_df
    .groupby("case_id", dropna=False)
    .agg(
        refined_selection_reason=(
            "refined_selection_reason",
            lambda values: "; ".join(sorted(set(values))),
        )
    )
    .reset_index()
)

refined_visual_case_base_columns = [
    column for column in refined_visual_cases_raw_df.columns
    if column != "refined_selection_reason"
]

refined_visual_cases_df = (
    refined_visual_cases_raw_df[refined_visual_case_base_columns]
    .drop_duplicates(subset=["case_id"])
    .merge(refined_selection_reason_df, on="case_id", how="left", validate="one_to_one")
    .sort_values(["category", "mask_type", "case_id"])
    .reset_index(drop=True)
)

# Attach any existing Notebook 24 comparison figure if available.
refined_visual_cases_df = refined_visual_cases_df.merge(
    previous_visual_cases_df[
        [
            "case_id",
            "comparison_figure_path",
        ]
    ].drop_duplicates(subset=["case_id"]),
    on="case_id",
    how="left",
    validate="one_to_one",
)

refined_visual_cases_path = metrics_dir / "comparison_visual_cases_refined_opencv_lama_stable_diffusion_50.csv"
refined_visual_cases_df.to_csv(refined_visual_cases_path, index=False)

print("Raw refined visual selection rows:", len(refined_visual_cases_raw_df))
print("Unique refined visual cases:", len(refined_visual_cases_df))
print("Saved refined visual cases:", refined_visual_cases_path)

display(
    refined_visual_cases_df[
        [
            "case_id",
            "category",
            "mask_type",
            "refined_selection_reason",
            "overall_metric_vote",
            *[f"{model_name}_metric_wins" for model_name in model_names],
            "metric_ties",
            "comparison_figure_path",
        ]
    ]
)

Raw refined visual selection rows: 67
Unique refined visual cases: 62
Saved refined visual cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_visual_cases_refined_opencv_lama_stable_diffusion_50.csv


,case_id,category,mask_type,refined_selection_reason,overall_metric_vote,opencv_telea_metric_wins,lama_metric_wins,stable_diffusion_inpainting_metric_wins,metric_ties,comparison_figure_path
0,p035_loss_large,abstraction_surrealism,loss_large,refined_category_mask_representative,lama,0,6,0,0,NaN
1,p037_loss_large,abstraction_surrealism,loss_large,refined_majority_vote_stable_diffusion_inpaint...,stable_diffusion_inpainting,1,1,4,0,D:\Masters\FH\Thesis\painting-restoration-eval...
2,p040_loss_large,abstraction_surrealism,loss_large,old_vs_refined_vote_changed; refined_metric_di...,opencv_telea,3,2,1,0,D:\Masters\FH\Thesis\painting-restoration-eval...
3,p031_loss_small,abstraction_surrealism,loss_small,refined_majority_vote_lama,lama,0,6,0,0,D:\Masters\FH\Thesis\painting-restoration-eval...
4,p032_loss_small,abstraction_surrealism,loss_small,refined_majority_vote_lama,lama,0,6,0,0,D:\Masters\FH\Thesis\painting-restoration-eval...
...,...,...,...,...,...,...,...,...,...,...
57,p001_scratch_thin,portrait_figure,scratch_thin,refined_metric_disagreement,lama,1,5,0,0,D:\Masters\FH\Thesis\painting-restoration-eval...
58,p002_scratch_thin,portrait_figure,scratch_thin,refined_metric_disagreement,lama,1,5,0,0,D:\Masters\FH\Thesis\painting-restoration-eval...
59,p004_scratch_thin,portrait_figure,scratch_thin,old_vs_refined_vote_changed,tie_lama_opencv_telea,3,3,0,0,D:\Masters\FH\Thesis\painting-restoration-eval...
60,p006_scratch_thin,portrait_figure,scratch_thin,refined_category_mask_representative,lama,0,6,0,0,NaN


In [19]:
def resolve_project_path(path_value: str | Path) -> Path:
    path = Path(str(path_value))

    if path.is_absolute():
        return path

    return PROJECT_ROOT / path


def file_exists_or_blank(path_value) -> bool:
    if pd.isna(path_value):
        return False

    if str(path_value).strip() == "":
        return False

    return resolve_project_path(path_value).exists()


def dataframe_to_html_table(
    df: pd.DataFrame,
    *,
    max_rows: int = 100,
    float_precision: int = 4,
) -> str:
    display_df = df.head(max_rows).copy()

    for column in display_df.select_dtypes(include=["float", "float64", "float32"]).columns:
        display_df[column] = display_df[column].round(float_precision)

    return display_df.to_html(index=False, escape=True, classes="data-table")


def image_to_base64_data_uri(path_value: str | Path, *, max_width: int = 1600) -> str:
    path = resolve_project_path(path_value)

    if not path.exists():
        raise FileNotFoundError(f"Missing image for report embedding: {path}")

    image = Image.open(path).convert("RGB")

    if image.width > max_width:
        scale = max_width / image.width
        new_size = (max_width, int(image.height * scale))
        image = image.resize(new_size)

    from io import BytesIO

    buffer = BytesIO()
    image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return f"data:image/png;base64,{encoded}"


available_refined_visual_cases_df = refined_visual_cases_df[
    refined_visual_cases_df["comparison_figure_path"].apply(file_exists_or_blank)
].copy()

missing_visual_figure_cases_df = refined_visual_cases_df[
    ~refined_visual_cases_df["comparison_figure_path"].apply(file_exists_or_blank)
].copy()

print("Refined visual cases with existing figures:", len(available_refined_visual_cases_df))
print("Refined visual cases without existing figures:", len(missing_visual_figure_cases_df))

if len(missing_visual_figure_cases_df) > 0:
    display(
        missing_visual_figure_cases_df[
            [
                "case_id",
                "category",
                "mask_type",
                "refined_selection_reason",
                "comparison_figure_path",
            ]
        ]
    )

Refined visual cases with existing figures: 17
Refined visual cases without existing figures: 45


,case_id,category,mask_type,refined_selection_reason,comparison_figure_path
0,p035_loss_large,abstraction_surrealism,loss_large,refined_category_mask_representative,NaN
5,p034_loss_small,abstraction_surrealism,loss_small,refined_category_mask_representative,NaN
6,p033_mixed_damage,abstraction_surrealism,mixed_damage,refined_category_mask_representative,NaN
8,p039_scratch_thin,abstraction_surrealism,scratch_thin,old_vs_refined_vote_changed,NaN
10,p024_loss_large,architecture_structured,loss_large,refined_metric_disagreement,NaN
12,p021_loss_small,architecture_structured,loss_small,refined_category_mask_representative,NaN
13,p026_loss_small,architecture_structured,loss_small,old_vs_refined_vote_changed,NaN
14,p027_loss_small,architecture_structured,loss_small,old_vs_refined_vote_changed,NaN
15,p030_loss_small,architecture_structured,loss_small,old_vs_refined_vote_changed,NaN
16,p021_mixed_damage,architecture_structured,mixed_damage,refined_category_mask_representative,NaN


In [20]:
def render_metric_card(title: str, value: str, subtitle: str = "") -> str:
    return f"""
    <div class="metric-card">
        <div class="metric-title">{title}</div>
        <div class="metric-value">{value}</div>
        <div class="metric-subtitle">{subtitle}</div>
    </div>
    """


def render_refined_visual_gallery(
    df: pd.DataFrame,
    *,
    title: str,
    max_items: int = 24,
) -> str:
    if df.empty:
        return f"""
        <section>
            <h2>{title}</h2>
            <p>No visual cases available for this section.</p>
        </section>
        """

    gallery_items = []

    for _, row in df.head(max_items).iterrows():
        image_uri = image_to_base64_data_uri(
            row["comparison_figure_path"],
            max_width=1600,
        )

        metric_summary = "<br>".join(
            [
                f"{model_registry[model_name]['display_name']}: "
                f"{int(row[f'{model_name}_metric_wins'])} wins"
                for model_name in model_names
            ]
        )

        winner_summary = "<br>".join(
            [
                f"{metric_name}: {row[f'winner_{metric_name}']}"
                for metric_name in refined_metric_policy
            ]
        )

        gallery_items.append(
            f"""
            <div class="visual-item">
                <h4>{row['case_id']} | {row['category']} | {row['mask_type']}</h4>
                <p><strong>Refined selection:</strong> {row['refined_selection_reason']}</p>
                <p><strong>Refined overall vote:</strong> {row['overall_metric_vote']}</p>
                <p><strong>Metric wins:</strong><br>{metric_summary}</p>
                <details>
                    <summary>Metric winners</summary>
                    <p>{winner_summary}</p>
                </details>
                <img src="{image_uri}" alt="{row['case_id']}">
            </div>
            """
        )

    return f"""
    <section>
        <h2>{title}</h2>
        <div class="visual-gallery">
            {''.join(gallery_items)}
        </div>
    </section>
    """


print("Refined visual gallery helpers ready.")

Refined visual gallery helpers ready.


In [21]:
# Compact policy table for the report.
refined_policy_rows = []

for metric_name, metric_info in refined_metric_policy.items():
    refined_policy_rows.append(
        {
            "metric_name": metric_name,
            "metric_family": metric_info["family"],
            "final_local_region": metric_info["region"],
            "description": metric_info["description"],
        }
    )

refined_policy_df = pd.DataFrame(refined_policy_rows)

# SSIM validation display table.
ssim_issue_display_df = ssim_issue_df.copy()

# Old vs refined vote summary.
old_vs_refined_vote_summary_df = (
    old_vs_refined_df
    .groupby(
        [
            "old_overall_metric_vote",
            "refined_overall_metric_vote",
        ],
        dropna=False,
    )
    .agg(cases=("case_id", "count"))
    .reset_index()
    .sort_values("cases", ascending=False)
)

# Refined majority-vote table.
refined_majority_vote_df = refined_win_rates_df[
    refined_win_rates_df["metric_name"] == "overall_metric_vote"
].copy()

# Per-metric winner table.
refined_per_metric_winners_df = refined_win_rates_df[
    refined_win_rates_df["metric_name"] != "overall_metric_vote"
].copy()

# Most important changed cases.
changed_cases_for_report_df = old_vs_refined_df[
    old_vs_refined_df["overall_vote_changed"]
].copy()

print("Refined policy:")
display(refined_policy_df)

print("\nSSIM issue summary:")
display(ssim_issue_display_df)

print("\nOld-vs-refined vote summary:")
display(old_vs_refined_vote_summary_df)

print("\nRefined majority-vote summary:")
display(refined_majority_vote_df)

print("\nChanged vote cases:", len(changed_cases_for_report_df))
display(changed_cases_for_report_df.head(30))

Refined policy:


,metric_name,metric_family,final_local_region,description
0,mse_improvement,classical,masked_region,MSE improvement on sparse masked region
1,psnr_improvement,classical,masked_region,PSNR improvement on sparse masked region
2,ssim_improvement,classical,mask_bbox_crop,SSIM improvement on mask-bounding-box crop
3,lpips_improvement,lpips,mask_bbox_crop,LPIPS improvement on mask-bounding-box crop
4,clip_improvement,feature,mask_bbox_crop,CLIP similarity improvement on mask-bounding-b...
5,dinov2_improvement,feature,mask_bbox_crop,DINOv2 similarity improvement on mask-bounding...



SSIM issue summary:


,model_name,masked_region_rows,masked_region_valid_ssim_rows,masked_region_invalid_ssim_rows,mask_bbox_crop_rows,mask_bbox_crop_valid_ssim_rows,mask_bbox_crop_invalid_ssim_rows
0,opencv_telea,200,0,200,200,200,0
1,lama,200,0,200,200,200,0
2,stable_diffusion_inpainting,200,0,200,200,200,0



Old-vs-refined vote summary:


,old_overall_metric_vote,refined_overall_metric_vote,cases
0,lama,lama,155
1,lama,tie_lama_opencv_telea,22
2,opencv_telea,opencv_telea,20
3,opencv_telea,tie_lama_opencv_telea,1
4,stable_diffusion_inpainting,stable_diffusion_inpainting,1
5,tie_lama_opencv_telea,opencv_telea,1



Refined majority-vote summary:


,metric_name,metric_description,metric_region,winner,cases,case_fraction
17,overall_metric_vote,Per-case majority vote across six refined loca...,mixed_refined_regions,lama,155,0.775
18,overall_metric_vote,Per-case majority vote across six refined loca...,mixed_refined_regions,tie_lama_opencv_telea,23,0.115
19,overall_metric_vote,Per-case majority vote across six refined loca...,mixed_refined_regions,opencv_telea,21,0.105
20,overall_metric_vote,Per-case majority vote across six refined loca...,mixed_refined_regions,stable_diffusion_inpainting,1,0.005



Changed vote cases: 24


,case_id,category,mask_type,old_overall_metric_vote,refined_overall_metric_vote,refined_opencv_telea_metric_wins,refined_lama_metric_wins,refined_stable_diffusion_inpainting_metric_wins,refined_metric_ties,overall_vote_changed
2,p001_mixed_damage,portrait_figure,mixed_damage,lama,tie_lama_opencv_telea,3,3,0,0,True
15,p004_scratch_thin,portrait_figure,scratch_thin,lama,tie_lama_opencv_telea,3,3,0,0,True
36,p010_loss_large,portrait_figure,loss_large,lama,tie_lama_opencv_telea,3,3,0,0,True
37,p010_loss_small,portrait_figure,loss_small,lama,tie_lama_opencv_telea,3,3,0,0,True
39,p010_scratch_thin,portrait_figure,scratch_thin,lama,tie_lama_opencv_telea,3,3,0,0,True
56,p015_loss_large,landscape_natural,loss_large,lama,tie_lama_opencv_telea,3,3,0,0,True
57,p015_loss_small,landscape_natural,loss_small,lama,tie_lama_opencv_telea,3,3,0,0,True
62,p016_mixed_damage,landscape_natural,mixed_damage,lama,tie_lama_opencv_telea,3,3,0,0,True
63,p016_scratch_thin,landscape_natural,scratch_thin,lama,tie_lama_opencv_telea,3,3,0,0,True
64,p017_loss_large,landscape_natural,loss_large,lama,tie_lama_opencv_telea,3,3,0,0,True


In [22]:
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

overview_cards_html = "".join(
    [
        render_metric_card(
            "Refined local cases",
            str(len(refined_comparison_df)),
            "Non-zero damage cases",
        ),
        render_metric_card(
            "Refined metrics",
            str(len(refined_metric_policy)),
            "MSE, PSNR, SSIM, LPIPS, CLIP, DINOv2",
        ),
        render_metric_card(
            "SSIM retained",
            "Yes",
            "Moved to mask_bbox_crop",
        ),
        render_metric_card(
            "Old-vs-refined vote changes",
            str(int(old_vs_refined_df["overall_vote_changed"].sum())),
            "Cases where majority vote changed",
        ),
        render_metric_card(
            "Refined disagreement cases",
            str(len(refined_metric_disagreement_cases_df)),
            "Cases where metric winners differ",
        ),
        render_metric_card(
            "Visual cases with figures",
            str(len(available_refined_visual_cases_df)),
            "Reused comparison grids from Notebook 24",
        ),
    ]
)

model_cards_html = "".join(
    [
        render_metric_card(
            model_registry[model_name]["display_name"],
            str(int(refined_comparison_df[f"{model_name}_metric_wins"].sum())),
            "Total refined metric wins across 200 local cases × 6 metrics",
        )
        for model_name in model_names
    ]
)

majority_vote_gallery_df = available_refined_visual_cases_df[
    available_refined_visual_cases_df["refined_selection_reason"].str.contains(
        "refined_majority_vote",
        na=False,
    )
].copy()

disagreement_gallery_df = available_refined_visual_cases_df[
    available_refined_visual_cases_df["refined_selection_reason"].str.contains(
        "refined_metric_disagreement",
        na=False,
    )
].copy()

category_mask_gallery_df = available_refined_visual_cases_df[
    available_refined_visual_cases_df["refined_selection_reason"].str.contains(
        "refined_category_mask_representative",
        na=False,
    )
].copy()

changed_vote_gallery_df = available_refined_visual_cases_df[
    available_refined_visual_cases_df["refined_selection_reason"].str.contains(
        "old_vs_refined_vote_changed",
        na=False,
    )
].copy()

html_report = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Refined Metric Region Policy Report</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 32px;
    background: #f7f7f7;
    color: #222;
    line-height: 1.45;
}}
h1, h2, h3 {{
    color: #111;
}}
section {{
    background: white;
    padding: 22px;
    margin: 26px 0;
    border-radius: 10px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.12);
}}
.metric-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(230px, 1fr));
    gap: 14px;
    margin: 20px 0;
}}
.metric-card {{
    background: #ffffff;
    border-left: 5px solid #444;
    padding: 14px;
    border-radius: 8px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.12);
}}
.metric-title {{
    font-size: 13px;
    color: #666;
    text-transform: uppercase;
}}
.metric-value {{
    font-size: 28px;
    font-weight: bold;
    margin: 8px 0;
}}
.metric-subtitle {{
    font-size: 13px;
    color: #777;
}}
.data-table {{
    border-collapse: collapse;
    width: 100%;
    margin: 12px 0 24px 0;
    font-size: 12px;
}}
.data-table th, .data-table td {{
    border: 1px solid #ddd;
    padding: 6px;
    text-align: left;
    vertical-align: top;
}}
.data-table th {{
    background: #efefef;
}}
.visual-gallery {{
    display: flex;
    flex-direction: column;
    gap: 28px;
}}
.visual-item {{
    border-top: 1px solid #ddd;
    padding-top: 18px;
}}
.visual-item img {{
    max-width: 100%;
    height: auto;
    border: 1px solid #ccc;
}}
.warning {{
    background: #fff7e6;
    border-left: 5px solid #cc8800;
    padding: 14px;
    border-radius: 8px;
}}
.note {{
    background: #eef5ff;
    border-left: 5px solid #3366aa;
    padding: 14px;
    border-radius: 8px;
}}
code {{
    background: #f0f0f0;
    padding: 2px 4px;
    border-radius: 4px;
}}
</style>
</head>
<body>

<h1>Refined Metric Region Policy Report</h1>
<p><strong>Generated:</strong> {generated_at}</p>

<section>
    <h2>Report purpose</h2>
    <p>
        This report refines the metric-region policy used for the final three-model comparison
        between OpenCV Telea, LaMa, and Stable Diffusion Inpainting.
    </p>
    <p>
        The initial comparison used sparse <code>masked_region</code> values for MSE, PSNR,
        and SSIM. The initial report revealed that sparse masked-region SSIM produced
        invalid local comparison values. SSIM is therefore retained, but moved to
        <code>mask_bbox_crop</code> for final local comparison.
    </p>
    <div class="note">
        This is a comparison-level refinement. The individual model metric files and model-level
        reports remain valid. This notebook rebuilds the final comparison tables using a corrected
        local metric-region policy.
    </div>
</section>

<section>
    <h2>Overview</h2>
    <div class="metric-grid">
        {overview_cards_html}
    </div>
</section>

<section>
    <h2>Refined metric-region policy</h2>
    <p>
        MSE and PSNR remain on the sparse masked region because they are direct pixel-error metrics.
        SSIM is moved to the mask-bounding-box crop because it requires an image-like local
        neighborhood with spatial context.
    </p>
    {dataframe_to_html_table(refined_policy_df, max_rows=20)}
</section>

<section>
    <h2>SSIM validation evidence</h2>
    <p>
        The table below summarizes valid and invalid SSIM values for sparse
        <code>masked_region</code> and image-like <code>mask_bbox_crop</code> regions.
    </p>
    {dataframe_to_html_table(ssim_issue_display_df, max_rows=20)}
</section>

<section>
    <h2>Total refined metric wins by model</h2>
    <div class="metric-grid">
        {model_cards_html}
    </div>
</section>

<section>
    <h2>Refined win-rate summary</h2>
    <p>
        Each non-zero damage case is evaluated across six refined local metrics.
        The model with the highest improvement value is counted as the metric winner.
    </p>
    {dataframe_to_html_table(refined_win_rates_df, max_rows=100)}
</section>

<section>
    <h2>Refined majority-vote summary</h2>
    <p>
        The overall metric vote is calculated per case from the six refined metric winners.
    </p>
    {dataframe_to_html_table(refined_majority_vote_df, max_rows=20)}
</section>

<section>
    <h2>Old-vs-refined comparison</h2>
    <p>
        This table compares the initial majority-vote result from Notebook 24 with the refined
        majority vote after moving SSIM to <code>mask_bbox_crop</code>.
    </p>
    {dataframe_to_html_table(old_vs_refined_vote_summary_df, max_rows=50)}
</section>

<section>
    <h2>Cases where majority vote changed</h2>
    <p>
        These cases changed majority-vote winner after the refined metric-region policy was applied.
        This section is included because silently changing comparison logic is how projects become
        cursed artifacts.
    </p>
    {dataframe_to_html_table(
        changed_cases_for_report_df[
            [
                "case_id",
                "category",
                "mask_type",
                "old_overall_metric_vote",
                "refined_overall_metric_vote",
                *[f"refined_{model_name}_metric_wins" for model_name in model_names],
                "refined_metric_ties",
            ]
        ],
        max_rows=100,
    )}
</section>

<section>
    <h2>Refined summary by mask type</h2>
    {dataframe_to_html_table(refined_summary_by_mask_type_df, max_rows=20)}
</section>

<section>
    <h2>Refined summary by painting category</h2>
    {dataframe_to_html_table(refined_summary_by_category_df, max_rows=20)}
</section>

<section>
    <h2>Refined metric-disagreement cases</h2>
    <p>
        These cases show disagreement between refined metric families or model winners.
        Metric disagreement is retained as diagnostic evidence rather than treated as an error.
    </p>
    {dataframe_to_html_table(
        refined_metric_disagreement_cases_df[
            [
                "case_id",
                "category",
                "mask_type",
                "overall_metric_vote",
                "num_unique_metric_winners",
                "disagreement_strength",
                *winner_columns,
                *[f"{model_name}_metric_wins" for model_name in model_names],
                "metric_ties",
            ]
        ],
        max_rows=80,
    )}
</section>

{render_refined_visual_gallery(
    majority_vote_gallery_df,
    title="Refined majority-vote visual examples",
    max_items=18,
)}

{render_refined_visual_gallery(
    disagreement_gallery_df,
    title="Refined metric-disagreement visual examples",
    max_items=18,
)}

{render_refined_visual_gallery(
    changed_vote_gallery_df,
    title="Old-vs-refined changed-vote visual examples",
    max_items=18,
)}

{render_refined_visual_gallery(
    category_mask_gallery_df,
    title="Refined category/mask representative visual examples",
    max_items=24,
)}

<section>
    <h2>Interpretation</h2>
    <p>
        The refined comparison retains SSIM as an important structural-similarity metric, but
        applies it on a region compatible with its assumptions. Sparse masked-region SSIM is
        not used for final majority voting because it does not provide a stable image-like
        local neighborhood.
    </p>
    <p>
        This correction improves methodological consistency while preserving the core comparison
        structure across OpenCV Telea, LaMa, and Stable Diffusion Inpainting.
    </p>
    <div class="warning">
        Model wins should still be interpreted as metric behavior under controlled synthetic
        damage, not as proof of conservation-grade restoration quality.
    </div>
</section>

</body>
</html>
"""

refined_report_path.write_text(html_report, encoding="utf-8")

print("Saved refined metric comparison report:")
print(refined_report_path)
print("Report characters:", len(html_report))

Saved refined metric comparison report:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html
Report characters: 25873995


In [23]:
expected_refined_outputs = [
    output_paths["refined_unified"],
    output_paths["refined_win_rates"],
    output_paths["refined_summary_by_mask_type"],
    output_paths["refined_summary_by_category"],
    output_paths["refined_metric_disagreement_cases"],
    output_paths["old_vs_refined"],
    refined_visual_cases_path,
    refined_report_path,
]

for output_file in expected_refined_outputs:
    if not output_file.exists():
        raise FileNotFoundError(f"Missing expected refined output: {output_file}")

saved_refined_unified_df = pd.read_csv(output_paths["refined_unified"])
saved_refined_win_rates_df = pd.read_csv(output_paths["refined_win_rates"])
saved_refined_summary_by_mask_type_df = pd.read_csv(output_paths["refined_summary_by_mask_type"])
saved_refined_summary_by_category_df = pd.read_csv(output_paths["refined_summary_by_category"])
saved_refined_metric_disagreement_df = pd.read_csv(output_paths["refined_metric_disagreement_cases"])
saved_old_vs_refined_df = pd.read_csv(output_paths["old_vs_refined"])
saved_refined_visual_cases_df = pd.read_csv(refined_visual_cases_path)

if len(saved_refined_unified_df) != 200:
    raise ValueError(
        f"Expected 200 refined unified rows, found {len(saved_refined_unified_df)}."
    )

if len(saved_refined_summary_by_mask_type_df) != 4:
    raise ValueError(
        f"Expected 4 refined mask-type summary rows, "
        f"found {len(saved_refined_summary_by_mask_type_df)}."
    )

if len(saved_refined_summary_by_category_df) != 5:
    raise ValueError(
        f"Expected 5 refined category summary rows, "
        f"found {len(saved_refined_summary_by_category_df)}."
    )

if len(saved_refined_win_rates_df) == 0:
    raise ValueError("Refined win-rate table is empty.")

if len(saved_refined_visual_cases_df) == 0:
    raise ValueError("Refined visual cases table is empty.")

if len(saved_old_vs_refined_df) != 200:
    raise ValueError(
        f"Expected 200 old-vs-refined rows, found {len(saved_old_vs_refined_df)}."
    )

required_refined_columns = [
    "case_id",
    "painting_id",
    "mask_id",
    "mask_type",
    "category",
    "title",
    "overall_metric_vote",
    "metric_ties",
    "mixed_metric_outcome",
]

for model_name in model_names:
    required_refined_columns.extend(
        [
            f"{model_name}_metric_wins",
            f"{model_name}_mse_improvement",
            f"{model_name}_psnr_improvement",
            f"{model_name}_ssim_improvement",
            f"{model_name}_lpips_improvement",
            f"{model_name}_clip_improvement",
            f"{model_name}_dinov2_improvement",
        ]
    )

for metric_name in refined_metric_policy:
    required_refined_columns.append(f"winner_{metric_name}")

missing_refined_columns = [
    column for column in required_refined_columns
    if column not in saved_refined_unified_df.columns
]

if missing_refined_columns:
    raise ValueError(
        f"Refined unified comparison missing columns: {missing_refined_columns}"
    )

expected_mask_counts = {
    "loss_large": 50,
    "loss_small": 50,
    "mixed_damage": 50,
    "scratch_thin": 50,
}

actual_mask_counts = saved_refined_unified_df["mask_type"].value_counts().to_dict()

print("Expected refined mask counts:", expected_mask_counts)
print("Actual refined mask counts:", actual_mask_counts)

for mask_type, expected_count in expected_mask_counts.items():
    actual_count = actual_mask_counts.get(mask_type, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Refined comparison mask type {mask_type!r}: "
            f"expected {expected_count}, found {actual_count}."
        )

if "zero_control" in actual_mask_counts:
    raise ValueError("Refined local comparison should not include zero_control rows.")

expected_category_counts = {
    "portrait_figure": 40,
    "landscape_natural": 40,
    "architecture_structured": 40,
    "abstraction_surrealism": 40,
    "high_texture_brushwork": 40,
}

actual_category_counts = saved_refined_unified_df["category"].value_counts().to_dict()

print("\nExpected refined category counts:", expected_category_counts)
print("Actual refined category counts:", actual_category_counts)

for category, expected_count in expected_category_counts.items():
    actual_count = actual_category_counts.get(category, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Refined comparison category {category!r}: "
            f"expected {expected_count}, found {actual_count}."
        )

allowed_winners = set(model_names)
allowed_winner_prefixes = {"tie_", "tie_all", "no_valid_values"}

for metric_name in refined_metric_policy:
    winner_column = f"winner_{metric_name}"

    invalid_winners = []

    for winner_value in saved_refined_unified_df[winner_column].dropna().astype(str).unique():
        if winner_value in allowed_winners:
            continue

        if winner_value in allowed_winner_prefixes:
            continue

        if winner_value.startswith("tie_"):
            continue

        invalid_winners.append(winner_value)

    if invalid_winners:
        raise ValueError(
            f"Invalid winner values in {winner_column}: {invalid_winners}"
        )

# Confirm refined SSIM is no longer globally invalid.
ssim_winner_values = set(saved_refined_unified_df["winner_ssim_improvement"].dropna().astype(str))

if ssim_winner_values == {"no_valid_values"}:
    raise ValueError(
        "Refined SSIM still produced only no_valid_values. "
        "SSIM region correction did not work."
    )

if "no_valid_values" in ssim_winner_values:
    raise ValueError(
        f"Refined SSIM still contains no_valid_values among winners: {ssim_winner_values}"
    )

required_visual_columns = [
    "case_id",
    "category",
    "mask_type",
    "refined_selection_reason",
    "overall_metric_vote",
    "comparison_figure_path",
]

missing_visual_columns = [
    column for column in required_visual_columns
    if column not in saved_refined_visual_cases_df.columns
]

if missing_visual_columns:
    raise ValueError(
        f"Refined visual cases missing columns: {missing_visual_columns}"
    )

html_content = refined_report_path.read_text(encoding="utf-8")

required_html_phrases = [
    "Refined Metric Region Policy Report",
    "Refined metric-region policy",
    "SSIM validation evidence",
    "Total refined metric wins by model",
    "Old-vs-refined comparison",
    "Refined metric-disagreement cases",
    "SSIM is therefore retained",
]

missing_html_phrases = [
    phrase for phrase in required_html_phrases
    if phrase not in html_content
]

if missing_html_phrases:
    raise ValueError(
        f"Refined HTML report missing expected phrases: {missing_html_phrases}"
    )

if len(html_content) < 20_000:
    raise ValueError(
        f"Refined HTML report seems too small: {len(html_content)} characters."
    )

embedded_image_count = html_content.count("data:image/png;base64,")

print("\nEmbedded image count:", embedded_image_count)

if embedded_image_count == 0:
    raise ValueError("Refined HTML report does not contain embedded images.")

print("\nSaved refined unified rows:", len(saved_refined_unified_df))
print("Saved refined win-rate rows:", len(saved_refined_win_rates_df))
print("Saved refined visual cases:", len(saved_refined_visual_cases_df))
print("Saved refined metric-disagreement rows:", len(saved_refined_metric_disagreement_df))
print("Old-vs-refined vote changes:", int(saved_old_vs_refined_df["overall_vote_changed"].sum()))
print("Refined report path:", refined_report_path)
print("Refined report characters:", len(html_content))
print("Final refined metric-region policy gates passed.")

Expected refined mask counts: {'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50}
Actual refined mask counts: {'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50}

Expected refined category counts: {'portrait_figure': 40, 'landscape_natural': 40, 'architecture_structured': 40, 'abstraction_surrealism': 40, 'high_texture_brushwork': 40}
Actual refined category counts: {'portrait_figure': 40, 'landscape_natural': 40, 'architecture_structured': 40, 'abstraction_surrealism': 40, 'high_texture_brushwork': 40}

Embedded image count: 22

Saved refined unified rows: 200
Saved refined win-rate rows: 21
Saved refined visual cases: 62
Saved refined metric-disagreement rows: 124
Old-vs-refined vote changes: 24
Refined report path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html
Refined report characters: 25873995
Final refined metric-region policy gates passed.


## Notebook 26 summary

This notebook refined the final local metric-region policy for the three-model comparison.

The initial comparison used sparse `masked_region` values for MSE, PSNR, and SSIM. The validation in this notebook confirmed that sparse masked-region SSIM is not suitable for final local model ranking.

SSIM was retained as an important structural-similarity metric, but moved to `mask_bbox_crop`, which provides an image-like local region with surrounding spatial context.

Final refined local metric-region policy:

- MSE improvement: `masked_region`
- PSNR improvement: `masked_region`
- SSIM improvement: `mask_bbox_crop`
- LPIPS improvement: `mask_bbox_crop`
- CLIP similarity improvement: `mask_bbox_crop`
- DINOv2 similarity improvement: `mask_bbox_crop`

This notebook rebuilt only the comparison-level outputs. It did not regenerate model-level metrics or model-level reports.

Main outputs:

- `outputs/metrics/comparison_unified_refined_opencv_lama_stable_diffusion_50.csv`
- `outputs/metrics/comparison_win_rates_refined_opencv_lama_stable_diffusion_50.csv`
- `outputs/metrics/comparison_summary_by_mask_type_refined_opencv_lama_stable_diffusion_50.csv`
- `outputs/metrics/comparison_summary_by_category_refined_opencv_lama_stable_diffusion_50.csv`
- `outputs/metrics/comparison_metric_disagreement_cases_refined_opencv_lama_stable_diffusion_50.csv`
- `outputs/metrics/comparison_old_vs_refined_metric_policy_50.csv`
- `outputs/metrics/comparison_visual_cases_refined_opencv_lama_stable_diffusion_50.csv`
- `outputs/reports/opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html`

The refined comparison keeps SSIM in the framework while aligning the region choice with the metric's structural assumptions.

This refined comparison should be used as the final comparison policy in the consolidated 50-painting evaluation report.